### 2. Libraries Used

First, let's import all the necessary libraries as specified in the document.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

### 3. Uploading the Dataset in Google Colab

Now, let's upload the dataset. Please select your CSV file when prompted.

In [ ]:
from google.colab import files

uploaded = files.upload()

### 4. Reading the CSV File

Next, we'll read the uploaded CSV file into a Pandas DataFrame and display its shape and the first few rows.

In [ ]:
filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("Dataset shape:", df.shape)
display(df.head())

### 5. Checking the Target Column

Let's check the column names and the distribution of the target variable `PlacementStatus`.

In [ ]:
print(df.columns.tolist())

print(df["PlacementStatus"].value_counts())

### 6. Separating Features and Target

Finally for this section, we'll separate the features (X) from the target variable (y).

In [ ]:
target = "PlacementStatus"

X = df.drop(columns=[target])
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

The initial data loading and preparation steps from the document are now set up. Let me know how you'd like to proceed with the next steps of the experiment, such as data splitting, scaling, and model training!

### 7. Data Splitting

Let's split the dataset into training and validation sets using `train_test_split`.

In [ ]:
# Split data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

### 8. Feature Scaling

Now, we'll apply `StandardScaler` to standardize the numerical features. It's important to fit the scaler only on the training data and then transform both training and validation sets.

In [ ]:
# Identify numerical columns for scaling (assuming all columns except StudentID and categorical ones are numerical)
numerical_cols = X_train.select_dtypes(include=np.number).columns.tolist()
# Exclude 'StudentID' if it's not a feature to be scaled
if 'StudentID' in numerical_cols:
    numerical_cols.remove('StudentID')

scaler = StandardScaler()

# Fit on training data and transform both training and validation data
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])

display(X_train.head())

### 9. Regularization Experiment Setup

Now, let's set up the experiment to compare Lasso (L1), Ridge (L2), and Elastic Net regularization. We will iterate through a range of `C` values and for each regularization type, train a `LogisticRegression` model, record its validation accuracy, and analyze its coefficients.

In [ ]:
C_values = [0.001, 0.01, 0.1, 1, 10, 100, 1000]

results = []

for C in C_values:
    # Lasso (L1 Regularization)
    print(f"\nProcessing Lasso (L1) with C={C}")
    lr_l1 = LogisticRegression(penalty='l1', C=C, solver='liblinear', random_state=42, max_iter=1000)
    lr_l1.fit(X_train, y_train)
    y_pred_l1 = lr_l1.predict(X_val)
    accuracy_l1 = accuracy_score(y_val, y_pred_l1)
    non_zero_coefs_l1 = np.sum(lr_l1.coef_ != 0)
    results.append({
        'Regularization': 'Lasso (L1)',
        'C': C,
        'Validation Accuracy': accuracy_l1,
        'Non-Zero Coefficients': non_zero_coefs_l1,
        'Coefficients': lr_l1.coef_.flatten().tolist()
    })

    # Ridge (L2 Regularization)
    print(f"Processing Ridge (L2) with C={C}")
    lr_l2 = LogisticRegression(penalty='l2', C=C, solver='liblinear', random_state=42, max_iter=1000)
    lr_l2.fit(X_train, y_train)
    y_pred_l2 = lr_l2.predict(X_val)
    accuracy_l2 = accuracy_score(y_val, y_pred_l2)
    # For Ridge, typically all coefficients are non-zero, but we count for consistency
    non_zero_coefs_l2 = np.sum(lr_l2.coef_ != 0)
    results.append({
        'Regularization': 'Ridge (L2)',
        'C': C,
        'Validation Accuracy': accuracy_l2,
        'Non-Zero Coefficients': non_zero_coefs_l2,
        'Coefficients': lr_l2.coef_.flatten().tolist()
    })

    # Elastic Net Regularization
    # Elastic Net requires 'saga' solver and l1_ratio parameter
    print(f"Processing Elastic Net with C={C}")
    lr_elasticnet = LogisticRegression(penalty='elasticnet', C=C, solver='saga', l1_ratio=0.5, random_state=42, max_iter=1000)
    lr_elasticnet.fit(X_train, y_train)
    y_pred_elasticnet = lr_elasticnet.predict(X_val)
    accuracy_elasticnet = accuracy_score(y_val, y_pred_elasticnet)
    non_zero_coefs_elasticnet = np.sum(lr_elasticnet.coef_ != 0)
    results.append({
        'Regularization': 'Elastic Net',
        'C': C,
        'Validation Accuracy': accuracy_elasticnet,
        'Non-Zero Coefficients': non_zero_coefs_elasticnet,
        'Coefficients': lr_elasticnet.coef_.flatten().tolist()
    })

# Convert results to DataFrame
results_df = pd.DataFrame(results)
display(results_df)


The experiment has been executed, and the results are now available in the `results_df` DataFrame. This DataFrame contains the validation accuracy, the number of non-zero coefficients, and the full list of coefficients for each combination of regularization type and C value. We can now proceed to visualize these results or perform further analysis as suggested in the document, such as plotting validation accuracy and observing coefficient shrinkage.

### 10. Visualizing Experiment Results

To analyze the impact of regularization, let's visualize the validation accuracy and the number of non-zero coefficients for different C values and regularization types.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot Validation Accuracy vs C
for penalty_type in results_df['Regularization'].unique():
    subset = results_df[results_df['Regularization'] == penalty_type]
    axes[0].plot(subset['C'], subset['Validation Accuracy'], label=penalty_type, marker='o')

axes[0].set_xscale('log')
axes[0].set_title('Validation Accuracy vs. C (Inverse of Regularization Strength)')
axes[0].set_xlabel('C (Log Scale)')
axes[0].set_ylabel('Validation Accuracy')
axes[0].legend()
axes[0].grid(True)

# Plot Non-Zero Coefficients vs C (mainly for L1 and Elastic Net)
for penalty_type in results_df['Regularization'].unique():
    subset = results_df[results_df['Regularization'] == penalty_type]
    # Only plot non-zero coefficients for L1 and Elastic Net as L2 typically doesn't yield zero coefficients
    if 'Lasso' in penalty_type or 'Elastic Net' in penalty_type:
        axes[1].plot(subset['C'], subset['Non-Zero Coefficients'], label=penalty_type, marker='o')

axes[1].set_xscale('log')
axes[1].set_title('Number of Non-Zero Coefficients vs. C')
axes[1].set_xlabel('C (Log Scale)')
axes[1].set_ylabel('Number of Non-Zero Coefficients')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

These plots illustrate the trade-offs between model complexity (number of non-zero coefficients) and performance (validation accuracy) as the regularization strength (`C`) changes. You can now analyze these graphs to identify optimal `C` values and compare the behavior of Lasso, Ridge, and Elastic Net.